# 05 · Export small JSON files for the website
The website reads only these files. Every number on it comes from notebooks 02–04.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "data"
WEB = ROOT / "web" / "public" / "data"
WEB.mkdir(parents=True, exist_ok=True)
results = json.loads((OUT / "results.json").read_text())
W0, W1 = results["window"]

def save(name, obj):
    path = WEB / f"{name}.json"
    path.write_text(json.dumps(obj, separators=(",", ":"), default=float))
    print(f"{name}.json  {path.stat().st_size / 1024:.1f} KB")

def r(x, d=0):
    return None if pd.isna(x) else round(float(x), d)

### Monthly series
One row per month: Nifty close and return, equity inflow, SIP contribution, and whether it was a fall month.

In [2]:
data = pd.read_parquet(OUT / "monthly.parquet")
data = data[(data["month"] >= W0)].reset_index(drop=True)
fall_months = {f["month"] for f in results["fall_months"]}
monthly = [{"month": m, "nifty": r(n, 0), "nifty_ret": r(nr, 2), "equity": r(e, 0), "sip": r(s, 0),
            "fall": m in fall_months, "in_test": W0 <= m <= W1}
           for m, n, nr, e, s in zip(data["month"], data["nifty_close"], data["nifty_return_pct"],
                                       data["equity_inflow_cr"], data["sip_cr"])]
save("monthly", monthly)
data.tail(3)

monthly.json  9.9 KB


,month,nifty_close,nifty_return_pct,equity_inflow_cr,equity_change_cr,equity_change_pct,sip_cr,sip_change_cr,sip_change_pct,sip_share_pct,equity_aum_cr
88,2026-08,24080.400391,-1.243456,29328.617931,4631.232540,18.751914,32297.0,336.0,1.051281,110.121111,3.921147e+06
89,2026-09,22620.449219,-6.062819,38306.552049,8977.934118,30.611514,32490.0,193.0,0.597579,84.815778,3.754275e+06
90,2026-10,23053.099609,1.912652,32993.637260,-5312.914789,-13.869468,32160.0,-330.0,-1.015697,97.473339,3.714550e+06


### The months around each fall

In [3]:
data = pd.read_parquet(OUT / "fall_windows.parquet")
falls = []
for m, g in data.groupby("fall_month"):
    falls.append({"month": m, "nifty_ret": r(g.loc[g["offset"] == 0, "nifty_return_pct"].iloc[0], 2),
                  "rows": [{"offset": int(o), "month": mm, "equity": r(e, 0), "sip": r(s, 0)}
                           for o, mm, e, s in zip(g["offset"], g["month"], g["equity_inflow_cr"], g["sip_cr"])]})
save("falls", falls)

falls.json  2.3 KB


### Where the money goes, by financial year (April to March)
Equity categories and fund groups, summed per financial year so the chart stays readable.

In [4]:
def fy(month):
    y, m = int(month[:4]), int(month[5:7])
    start = y if m >= 4 else y - 1
    return f"FY{str(start + 1)[2:]}"            # April 2025 to March 2026 is FY26

data = pd.read_parquet(OUT / "equity_categories.parquet")
data = data[(data["month"] >= W0) & (data["month"] <= W1)]
data["fy"] = data["month"].map(fy)
cats = data.groupby(["fy", "category"])["net_inflow_cr"].sum().round(0).reset_index()
months_in_fy = data.groupby("fy")["month"].nunique().to_dict()

data = pd.read_parquet(OUT / "groups.parquet")
data = data[(data["month"] >= W0) & (data["month"] <= W1)]
data["fy"] = data["month"].map(fy)
groups = data.groupby(["fy", "group"])["net_inflow_cr"].sum().round(0).reset_index()

save("categories", {
    "months_in_fy": months_in_fy,
    "equity": cats.rename(columns={"net_inflow_cr": "inflow"}).to_dict("records"),
    "groups": groups.rename(columns={"net_inflow_cr": "inflow"}).to_dict("records"),
})
cats.pivot(index="category", columns="fy", values="net_inflow_cr")

categories.json  7.0 KB


fy,FY20,FY21,FY22,FY23,FY24,FY25,FY26,FY27
category,,,,,,,,
ELSS (tax saver),8187.0,-149.0,1452.0,7744.0,1040.0,1306.0,-5858.0,-5130.0
Flexi cap,NaN,-9045.0,35877.0,16961.0,15502.0,49580.0,89213.0,42005.0
Large & mid cap,5292.0,393.0,14734.0,18221.0,22416.0,40308.0,44306.0,32144.0
Large cap,16003.0,-10587.0,13518.0,8373.0,-613.0,23487.0,24189.0,5608.0
Mid cap,13517.0,-3747.0,16309.0,20206.0,22226.0,41409.0,51197.0,42936.0
Multi cap,15560.0,-3655.0,28156.0,11420.0,22958.0,42282.0,33217.0,22965.0
Sectoral & thematic,4269.0,9801.0,27128.0,23731.0,46138.0,146656.0,29975.0,12462.0
Small cap,10431.0,-3035.0,10116.0,22104.0,40189.0,41673.0,51872.0,48620.0
"Value, contra, focused & dividend yield",10527.0,-5943.0,17114.0,17993.0,14235.0,30352.0,28424.0,14038.0


### Headline numbers and test results

In [5]:
m = pd.read_parquet(OUT / "monthly.parquet")
inside = m[(m["month"] >= W0) & (m["month"] <= W1)]
log = pd.read_csv(OUT / "extraction_log.csv")
latest = inside.iloc[-1]
overview = {
    "window": [W0, W1], "months": int(len(inside)),
    "latest": {"month": latest["month"], "equity": r(latest["equity_inflow_cr"]), "sip": r(latest["sip_cr"]),
               "nifty_ret": r(latest["nifty_return_pct"], 2)},
    "totals": {"equity": r(inside["equity_inflow_cr"].sum()), "sip": r(inside["sip_cr"].sum())},
    "outflow_months": int((inside["equity_inflow_cr"] < 0).sum()),
    "inflow_streak": int((inside["equity_inflow_cr"][::-1] > 0).cummin().sum()),   # months in a row of net inflows, up to the latest
    "sip_growth": {"from": r(inside["sip_cr"].iloc[0]), "to": r(inside["sip_cr"].iloc[-1])},
    "checks": {"run": int(log["checked"].sum()), "failed": int(log["failed"].sum())},
    "results": results,
}
save("overview", overview)
print(json.dumps({k: overview[k] for k in ("latest", "totals", "outflow_months", "inflow_streak", "sip_growth", "checks")}, indent=1))

overview.json  2.3 KB
{
 "latest": {
  "month": "2026-10",
  "equity": 32994.0,
  "sip": 32160.0,
  "nifty_ret": 1.91
 },
 "totals": {
  "equity": 1532307.0,
  "sip": 1537620.0
 },
 "outflow_months": 8,
 "inflow_streak": 68,
 "sip_growth": {
  "from": 8238.0,
  "to": 32160.0
 },
 "checks": {
  "run": 741,
  "failed": 0
 }
}


### Check: every file is small and matches the results

In [6]:
for f in sorted(WEB.glob("*.json")):
    assert f.stat().st_size < 1_000_000, f"{f.name} is too big"
check = json.loads((WEB / "overview.json").read_text())["results"]
assert check["h1"]["verdict"] == results["h1"]["verdict"] and check["h2"]["verdict"] == results["h2"]["verdict"]
print("all files under 1 MB; verdicts match results.json")

all files under 1 MB; verdicts match results.json
